# Solar Filament Segmentation — inference pipeline
Loads `weights/best.pt`, tunes post-processing on the validation set, writes `submission.csv`.
Set `CFG` paths before running. Runs on CPU or GPU (T4 recommended).

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import filament as F
import torch

print('device:', F.dev)
if F.dev == 'cuda':
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i))

## 1. (Optional) Train
Skip if `weights/best.pt` already exists.

In [ ]:
SKIP_TRAIN = os.path.exists(F.CFG['ckpt'])
print('skipping train — checkpoint found' if SKIP_TRAIN else 'training...')
if not SKIP_TRAIN:
    F.train()

## 2. Load model

In [ ]:
model = F.build(pretrained=False).to(F.dev)
model.load_state_dict(
    {k: v.float() for k, v in torch.load(F.CFG['ckpt'], map_location=F.dev).items()}
)
model.eval()
print('loaded', F.CFG['ckpt'])

## 3. Tune post-processing on validation set
Runs TTA on val images, then grid-searches `t_lo`, `t_hi`, `min_meanp`, `min_area`.
Updates `CFG` in place — skip if you want to use the defaults.

In [ ]:
import random
random.seed(F.CFG['seed'])

items = F.load_train_items()
files = sorted({it[0] for it in items})
random.shuffle(files)
val_files = set(files[:max(1, int(F.CFG['val_frac'] * len(files)))])
val_items = [it for it in items if it[0] in val_files]
print(f'{len(val_items)} val items from {len(val_files)} images')

probs, gts = F.collect_val_probs(model, val_items)
best = F.tune_postprocess(probs, gts)
F.CFG.update(best)
print('CFG updated:', {k: F.CFG[k] for k in ('t_lo','t_hi','min_meanp','min_area')})

## 4. Predict test set

In [ ]:
F.predict('submission.csv')

## 5. Validate submission format

In [ ]:
import glob, pandas as pd, numpy as np, re
from pycocotools import mask as mu

df = pd.read_csv('submission.csv')
test_stems = {F.stem(p) for p in glob.glob(F.CFG['test_dir'] + '/*')}

checks = {}
checks['header_ok']           = list(df.columns) == ['filament_id', 'segmentation_rle']
checks['ids_unique']          = df['filament_id'].is_unique
checks['ids_match_pattern']   = df['filament_id'].str.match(r'^[A-Za-z0-9]+_\d+$').all()
checks['all_stems_in_test']   = df['filament_id'].str.rsplit('_',n=1).str[0].isin(test_stems).all()
checks['complete_test_set']   = test_stems == set(df['filament_id'].str.rsplit('_',n=1).str[0])
checks['no_empty_masks']      = True
checks['decode_2048_ok']      = True

for _, row in df.iterrows():
    rle_obj = {'size': [2048, 2048], 'counts': row['segmentation_rle'].encode()}
    try:
        m = mu.decode(rle_obj)
        if m.sum() == 0: checks['no_empty_masks'] = False
    except Exception:
        checks['decode_2048_ok'] = False

for k, v in checks.items():
    print(f'  {k:<28} {v}')
print(f'\n{len(df)} rows, {df["filament_id"].str.rsplit("_",n=1).str[0].nunique()} images')